# Chapter 8 — Binary Search

*Built with [Claude](https://claude.ai) by Anthropic.*

Binary search halves the search space each step for O(log n) lookup. The key skill is recognizing when a problem's answer has a monotonic property — then you can binary search the answer space, not just an array.

In [12]:
import bisect
import numpy as np
from typing import List

def show_search(nums, left, right, mid, target):
    parts = []
    for i, v in enumerate(nums):
        s = f"{v:>4}"
        if i == mid:   s = f"[{v:>3}]"
        elif i == left:  s = f"L{v:>3}"
        elif i == right: s = f"{v:>3}R"
        parts.append(s)
    print(f"  target={target}  " + '  '.join(parts) + f"  l={left} m={mid} r={right}")

print('  Helpers loaded')

  Helpers loaded


---

## Part 1 — Classic Search on Sorted Array

**DS/ML connection:** `np.searchsorted`, `bisect.bisect_left` — binary search on sorted indices. Used for fast interval lookups, sorted feature quantiles, and bucket assignment in histogram encoders.

In [13]:
# ── DS/ML PARALLEL: numpy searchsorted ──
arr = np.array([1, 3, 5, 7, 9, 11, 13])
targets = [7, 6, 0, 14]
for t in targets:
    idx = np.searchsorted(arr, t)
    found = idx < len(arr) and arr[idx] == t
    print(f"  target={t:>3}  searchsorted→{idx}  found={found}")

  target=  7  searchsorted→3  found=True
  target=  6  searchsorted→3  found=False
  target=  0  searchsorted→0  found=False
  target= 14  searchsorted→7  found=False


### LC 704 — Binary Search

**Problem:** Given a sorted array and a target, return the index if found, else `-1`.

**DS parallel:** `np.searchsorted` — same operation, manual version helps you write custom variants.

**Key insight:** Maintain inclusive `[left, right]` bounds. At each step compute `mid`, compare, and narrow. Loop while `left <= right`.

In [14]:
# ── DS/ML APPROACH: numpy / bisect ──
def search_numpy(nums, target):
    arr = np.array(nums)
    idx = np.searchsorted(arr, target)
    return int(idx) if idx < len(arr) and arr[idx] == target else -1

def search_bisect(nums, target):
    idx = bisect.bisect_left(nums, target)
    return idx if idx < len(nums) and nums[idx] == target else -1

# ── RAW PYTHON (interview answer) ──
def search(nums: List[int], target: int) -> int:
    left, right = 0, len(nums) - 1
    while left <= right:
        mid = left + (right - left) // 2
        if nums[mid] == target:
            return mid
        elif nums[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    return -1

nums = [-1, 0, 3, 5, 9, 12]
for t in [9, 2]:
    print(f"  target={t}: numpy={search_numpy(nums, t)}  bisect={search_bisect(nums, t)}  manual={search(nums, t)}")

  target=9: numpy=4  bisect=4  manual=4
  target=2: numpy=-1  bisect=-1  manual=-1


In [15]:
# ── TRACE: LC 704 ──
def search_trace(nums, target):
    left, right = 0, len(nums) - 1
    step = 0
    print(f"  nums={nums}, target={target}")
    while left <= right:
        mid = left + (right - left) // 2
        show_search(nums, left, right, mid, target)
        if nums[mid] == target:
            print(f"  Found at index {mid}")
            return mid
        elif nums[mid] < target:
            print(f"  {nums[mid]} < {target}: move left to {mid+1}")
            left = mid + 1
        else:
            print(f"  {nums[mid]} > {target}: move right to {mid-1}")
            right = mid - 1
        step += 1
    print(f"  Not found (-1)")
    return -1

search_trace([-1, 0, 3, 5, 9, 12], 9)

  nums=[-1, 0, 3, 5, 9, 12], target=9
  target=9  L -1     0  [  3]     5     9   12R  l=0 m=2 r=5
  3 < 9: move left to 3
  target=9    -1     0     3  L  5  [  9]   12R  l=3 m=4 r=5
  Found at index 4


4

### LC 35 — Search Insert Position

**Problem:** Given a sorted array and target, return the index if found, else the index where it would be inserted.

**DS parallel:** `np.searchsorted(arr, target, side='left')` — bucket assignment in histogram encoders.

**Key insight:** Same as Template 1, but return `left` (not `-1`) when the target is not found. `left` ends up at the insertion point.

In [16]:
# ── DS/ML APPROACH: searchsorted ──
def searchInsert_numpy(nums, target):
    return int(np.searchsorted(nums, target, side='left'))

# ── RAW PYTHON (interview answer) ──
def searchInsert(nums: List[int], target: int) -> int:
    left, right = 0, len(nums) - 1
    while left <= right:
        mid = left + (right - left) // 2
        if nums[mid] == target:
            return mid
        elif nums[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    return left   # insertion point

nums = [1, 3, 5, 6]
for t in [5, 2, 7, 0]:
    np_res = searchInsert_numpy(nums, t)
    py_res = searchInsert(nums, t)
    print(f"  target={t}: numpy={np_res}  manual={py_res}")

  target=5: numpy=2  manual=2
  target=2: numpy=1  manual=1
  target=7: numpy=4  manual=4
  target=0: numpy=0  manual=0


---

## Part 2 — Modified Binary Search (Rotated Arrays)

**DS/ML connection:** When data has been sorted and then shifted (e.g., a circular buffer of timestamps), you need to identify which half is still sorted before you can eliminate it.

### LC 153 — Find Minimum in Rotated Sorted Array

**Problem:** A sorted array has been rotated. Find the minimum element in O(log n).

**DS parallel:** Finding the wrap-around point in a circular time-series buffer.

**Key insight:** Compare `nums[mid]` to `nums[hi]`. If `nums[mid] > nums[hi]`, the rotation point (and minimum) is in the right half. Otherwise it's in the left half (including `mid`).

In [17]:
# ── DS/ML APPROACH: numpy min (O(n)) ──
def findMin_numpy(nums):
    return int(np.min(nums))

# ── RAW PYTHON (interview answer) ──
def findMin(nums: List[int]) -> int:
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] > nums[hi]:
            lo = mid + 1    # min is in right half
        else:
            hi = mid        # min is in left half (including mid)
    return nums[lo]

tests = [[3,4,5,1,2], [4,5,6,7,0,1,2], [11,13,15,17]]
for nums in tests:
    print(f"  {nums}: numpy={findMin_numpy(nums)}  binary={findMin(nums)}")

  [3, 4, 5, 1, 2]: numpy=1  binary=1
  [4, 5, 6, 7, 0, 1, 2]: numpy=0  binary=0
  [11, 13, 15, 17]: numpy=11  binary=11


In [18]:
# ── TRACE: LC 153 ──
def findMin_trace(nums):
    lo, hi = 0, len(nums) - 1
    print(f"  nums={nums}")
    print(f"  {'lo':>4}  {'hi':>4}  {'mid':>4}  {'nums[mid]':>10}  {'nums[hi]':>9}  action")
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] > nums[hi]:
            action = f"lo → {mid+1}  (min in right)"
            lo = mid + 1
        else:
            action = f"hi → {mid}   (min in left)"
            hi = mid
        print(f"  {lo:>4}  {hi:>4}  {mid:>4}  {nums[mid]:>10}  {nums[hi]:>9}  {action}")
    print(f"  minimum = nums[{lo}] = {nums[lo]}")

findMin_trace([3,4,5,1,2])

  nums=[3, 4, 5, 1, 2]
    lo    hi   mid   nums[mid]   nums[hi]  action
     3     4     2           5          2  lo → 3  (min in right)
     3     3     3           1          1  hi → 3   (min in left)
  minimum = nums[3] = 1


---

## Part 3 — Binary Search on the Answer (Solution Space)

**DS/ML connection:** Threshold optimization — "find the minimum batch size such that training fits in memory," "find the minimum K neighbors such that validation accuracy exceeds 0.9." These are answer-space binary searches: the answer has a monotonic property, so you binary search it instead of trying every value.

### LC 875 — Koko Eating Bananas

**Problem:** Koko has `piles` of bananas. She can eat `k` bananas/hour and has `h` hours. Find the minimum `k` such that she finishes in time.

**DS parallel:** Minimum throughput needed to process a dataset in a deadline — the answer space is `[1, max(piles)]` and `feasible(k)` is monotonically decreasing.

**Key insight:** Binary search `k` in `[1, max(piles)]`. For each `k`, check if `sum(ceil(pile/k)) <= h`.

In [19]:
import math

# ── DS/ML APPROACH: linear scan (O(max(piles))) ──
def minEatingSpeed_linear(piles, h):
    for k in range(1, max(piles) + 1):
        if sum(math.ceil(p / k) for p in piles) <= h:
            return k
    return max(piles)

# ── RAW PYTHON (interview answer) ──
def minEatingSpeed(piles: List[int], h: int) -> int:
    def feasible(k):
        return sum(math.ceil(p / k) for p in piles) <= h

    left, right = 1, max(piles)
    while left < right:
        mid = (left + right) // 2
        if feasible(mid):
            right = mid       # mid works, try smaller
        else:
            left = mid + 1    # mid too slow
    return left

tests = [([3,6,7,11], 8), ([30,11,23,4,20], 5)]
for piles, h in tests:
    print(f"  piles={piles}, h={h}")
    print(f"    linear: {minEatingSpeed_linear(piles, h)}")
    print(f"    binary: {minEatingSpeed(piles, h)}")

  piles=[3, 6, 7, 11], h=8
    linear: 4
    binary: 4
  piles=[30, 11, 23, 4, 20], h=5
    linear: 30
    binary: 30


In [20]:
# ── TRACE: LC 875 ──
def minEatingSpeed_trace(piles, h):
    def feasible(k):
        hours = sum(math.ceil(p / k) for p in piles)
        return hours, hours <= h

    left, right = 1, max(piles)
    print(f"  piles={piles}, h={h}")
    print(f"  answer space: [{left}, {right}]")
    print(f"  {'left':>6}  {'right':>6}  {'mid':>5}  {'hours':>6}  {'feasible':>9}  action")
    while left < right:
        mid = (left + right) // 2
        hours, ok = feasible(mid)
        if ok:
            action = f"right → {mid}"
            right = mid
        else:
            action = f"left  → {mid+1}"
            left = mid + 1
        print(f"  {left:>6}  {right:>6}  {mid:>5}  {hours:>6}  {str(ok):>9}  {action}")
    print(f"  minimum k = {left}")

minEatingSpeed_trace([3,6,7,11], 8)

  piles=[3, 6, 7, 11], h=8
  answer space: [1, 11]
    left   right    mid   hours   feasible  action
       1       6      6       6       True  right → 6
       4       6      3      10      False  left  → 4
       4       5      5       8       True  right → 5
       4       4      4       8       True  right → 4
  minimum k = 4


### LC 1011 — Capacity to Ship Packages Within D Days

**Problem:** Given package `weights` in order and `days` deadline, find the minimum ship capacity that ships all packages in time.

**DS parallel:** Minimum batch size to process all records within a deadline — the answer space is `[max(weights), sum(weights)]`.

**Key insight:** Binary search capacity in `[max(weights), sum(weights)]`. `feasible(cap)` = check if you can ship all packages in ≤ `days` days with capacity `cap`.

In [21]:
# ── DS/ML APPROACH: linear scan ──
def shipWithinDays_linear(weights, days):
    for cap in range(max(weights), sum(weights) + 1):
        curr, d = 0, 1
        for w in weights:
            if curr + w > cap:
                d += 1
                curr = 0
            curr += w
        if d <= days:
            return cap
    return sum(weights)

# ── RAW PYTHON (interview answer) ──
def shipWithinDays(weights: List[int], days: int) -> int:
    def feasible(cap):
        curr, d = 0, 1
        for w in weights:
            if curr + w > cap:
                d += 1
                curr = 0
            curr += w
        return d <= days

    left, right = max(weights), sum(weights)
    while left < right:
        mid = (left + right) // 2
        if feasible(mid):
            right = mid
        else:
            left = mid + 1
    return left

tests = [([1,2,3,4,5,6,7,8,9,10], 5), ([3,2,2,4,1,4], 3)]
for weights, days in tests:
    print(f"  weights={weights}, days={days}")
    print(f"    linear: {shipWithinDays_linear(weights, days)}")
    print(f"    binary: {shipWithinDays(weights, days)}")

  weights=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10], days=5
    linear: 15
    binary: 15
  weights=[3, 2, 2, 4, 1, 4], days=3
    linear: 6
    binary: 6


In [22]:
# ── TRACE: LC 1011 ──
def shipWithinDays_trace(weights, days):
    def feasible(cap):
        curr, d = 0, 1
        for w in weights:
            if curr + w > cap:
                d += 1; curr = 0
            curr += w
        return d, d <= days

    left, right = max(weights), sum(weights)
    print(f"  weights={weights}, days={days}")
    print(f"  answer space: [{left}, {right}]")
    print(f"  {'left':>6}  {'right':>6}  {'mid':>5}  {'days_used':>10}  {'ok':>4}  action")
    while left < right:
        mid = (left + right) // 2
        d_used, ok = feasible(mid)
        if ok:
            action = f"right → {mid}"
            right = mid
        else:
            action = f"left → {mid+1}"
            left = mid + 1
        print(f"  {left:>6}  {right:>6}  {mid:>5}  {d_used:>10}  {str(ok):>4}  {action}")
    print(f"  minimum capacity = {left}")

shipWithinDays_trace([1,2,3,4,5,6,7,8,9,10], 5)

  weights=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10], days=5
  answer space: [10, 55]
    left   right    mid   days_used    ok  action
      10      32     32           2  True  right → 32
      10      21     21           3  True  right → 21
      10      15     15           5  True  right → 15
      13      15     12           6  False  left → 13
      15      15     14           6  False  left → 15
  minimum capacity = 15
